# Demo guiada — una función de predicción y una interfaz Streamlit

Esta libreta prepara la práctica de Streamlit sin importar módulos de solución. Primero probamos la regla de negocio; después pensamos cómo conectarla a una pantalla.


## 1. Contrato antes de pantalla

La UI no debe inventar reglas: recoge valores, llama a una función y muestra el resultado.

| Entrada | Tipo | Rango orientativo |
| --- | --- | --- |
| `tenure_months` | int | 0-72 |
| `monthly_charges` | float | 0-150 |
| `support_tickets` | int | 0-10 |
| `has_contract` | bool | true/false |


In [ ]:
def predict_churn(tenure_months: int, monthly_charges: float, support_tickets: int, has_contract: bool) -> dict:
    if not 0 <= tenure_months <= 72:
        raise ValueError("tenure_months fuera de rango")
    if not 0 <= monthly_charges <= 150:
        raise ValueError("monthly_charges fuera de rango")
    if not 0 <= support_tickets <= 10:
        raise ValueError("support_tickets fuera de rango")

    score = 0.20
    score += 0.25 if tenure_months < 6 else 0.0
    score += 0.25 if monthly_charges > 80 else 0.0
    score += 0.20 if support_tickets >= 3 else 0.0
    score -= 0.20 if has_contract else 0.0
    score = min(max(score, 0.05), 0.95)
    return {"risk_score": round(score, 2), "label": "high" if score >= 0.50 else "low"}

profiles = {
    "riesgo_alto": (2, 95, 4, False),
    "riesgo_bajo": (36, 35, 0, True),
    "inicial": (12, 60, 1, False),
}
results = {name: predict_churn(*values) for name, values in profiles.items()}
display(results)
assert results["riesgo_alto"]["risk_score"] > results["riesgo_bajo"]["risk_score"]

### Predice antes de ejecutar

La libreta incluye un perfil base para que la celda se ejecute desde cero.

**TODO 1:** añade un segundo perfil en `profiles` con valores extremos o normales elegidos por ti. Importa porque una demo debe enseñar el comportamiento de la regla ante casos distintos. Inspecciona la función `predict_churn`. Verifica que todos los scores quedan entre 0.05 y 0.95.


In [ ]:
profiles = [
    (6, 85, 2, False),  # perfil base para ejecutar la libreta
    # TODO 1: añade aquí un segundo perfil propio, por ejemplo cambiando contrato, tickets o meses.
]

results = [predict_churn(*profile) for profile in profiles]
for profile, result in zip(profiles, results):
    print(profile, result)
    assert 0.05 <= result["risk_score"] <= 0.95


## 2. Modelo mental de Streamlit

```text
rerun → dibujar formulario → ¿submitted?
  ├─ no: mostrar estado inicial
  └─ sí: validar inputs → llamar predict_churn → mostrar resultado o error
```

**TODO 2:** identifica qué parte vive en `model.py` y qué parte vive en `app.py`. Importa porque mezclar UI y predicción hace difícil probar. Inspecciona el patrón formulario/submit de Streamlit. Verifica que puedes probar `predict_churn` sin abrir la web.


In [ ]:
def render_message(result: dict) -> str:
    if result["label"] == "high":
        return f"Riesgo alto ({result['risk_score']:.0%}). Revisa acciones de retención."
    return f"Riesgo bajo ({result['risk_score']:.0%}). Mantén seguimiento normal."

message = render_message(my_result)
print(message)
assert "Riesgo" in message

## 3. Preparar la práctica

En la práctica, `app.py` debe hacer sólo tres cosas:

1. definir controles de entrada claros;
2. llamar a `predict_churn` al enviar;
3. mostrar resultado y errores de forma comprensible.

No ejecutes Streamlit desde esta libreta con subprocess o CLI local; en Databricks revisaremos el código y la ejecución se hará en el entorno indicado por el profesor.


## Cierre

Transfiere el patrón, no los datos: contrato pequeño, función probada, UI fina y mensajes entendibles para usuarios no técnicos.
